1. Downlaod the data

In [1]:
from pickletools import optimize

from torch.xpu import device

from dataset.data import download_udacity_dataset
download_udacity_dataset()

Dataset already exists. Skipping download.


2. preprocessing

In [2]:
import pandas as pd
#Shape
df = pd.read_csv('data/training_frames_keypoints.csv')
print(df.shape)
print(df.head(2))
print("Total Nan:",df.isnull().sum().sum())
print(df.isnull().sum()[df.isnull().sum()>0])

(3462, 137)
              Unnamed: 0     0     1     2      3     4      5     6      7  \
0      Luis_Fonsi_21.jpg  45.0  98.0  47.0  106.0  49.0  110.0  53.0  119.0   
1  Lincoln_Chafee_52.jpg  41.0  83.0  43.0   91.0  45.0  100.0  47.0  108.0   

      8  ...   126    127   128    129   130    131   132    133   134    135  
0  56.0  ...  83.0  119.0  90.0  117.0  83.0  119.0  81.0  122.0  77.0  122.0  
1  51.0  ...  85.0  122.0  94.0  120.0  85.0  122.0  83.0  122.0  79.0  122.0  

[2 rows x 137 columns]
Total Nan: 0
Series([], dtype: int64)


load the data and do preprocessing

In [3]:
from src.data_pipeline import FacialKeypointsDataset
csv_path = 'data/training_frames_keypoints.csv' #label
img_dir = 'data/training' #feature
dataset = FacialKeypointsDataset(csv_file=csv_path, root_dir=img_dir) #preprocessing
print("preprocessing done")

preprocessing done


In [4]:
from torch.utils.data import DataLoader #knowing the shape
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)
batch = next(iter(loader))
images, keypoints, masks = batch['image'], batch['keypoints'], batch['mask']
print(f"images.shape: {images.shape}, keypoints.shape: {keypoints.shape}, masks.shape: {masks.shape}")

images.shape: torch.Size([32, 1, 96, 96]), keypoints.shape: torch.Size([32, 90, 2]), masks.shape: torch.Size([32, 90])


In [5]:
#convert the photo to .pt file
import torch
all_images = []
all_keypoints = []
all_masks = []
for batch in loader:
    all_images.append(batch['image'])
    all_keypoints.append(batch['keypoints'])
    all_masks.append(batch['mask'])
all_images = torch.cat(all_images, dim=0)
all_keypoints = torch.cat(all_keypoints, dim=0)
all_masks = torch.cat(all_masks, dim=0)
torch.save(all_images, 'data/all_images.pt')
torch.save(all_keypoints, 'data/all_keypoints.pt')
torch.save(all_masks, 'data/all_masks.pt')
print("saved all_images.pt, all_keypoints.pt, all_masks.pt")

saved all_images.pt, all_keypoints.pt, all_masks.pt


Train the model

In [7]:
import torch
from models.naimishnet import NaimishNet

#Gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
images_gpu = all_images.to(device)
keypoints_gpu = all_keypoints.to(device)
masks_gpu = all_masks.to(device)

#Optimizer and loss function
model = NaimishNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss(reduce='none') #loss function

batch_size = 32
num_samples = images_gpu.size(0)

for epoch in range(20):
    model.train()
    running_loss = 0.0
    for i in range(0, num_samples, batch_size):
        inputs = images_gpu[i:i+batch_size]
        targets = keypoints_gpu[i:i+batch_size]
        masks = masks_gpu[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(inputs)
        outputs = outputs.view(-1, 90, 2) #reshape to (batch_size, 90, 2)

        loss = criterion(outputs, targets)
        masked_loss = loss * masks.unsqueeze(-1)
        mean_loss = masked_loss.sum() / masks.sum()


        mean_loss.backward()
        optimizer.step()

        running_loss += mean_loss.item()

    print(f"Epoch [{epoch+1}/20], Loss: {running_loss:.4f}")

torch.save(model.state_dict(), 'output/naimishnet.pth')
print("Model saved to output/naimishnet.pth")


/home/hinze/.pyenv/versions/3.12.7/lib/python3.12/site-packages/torch/nn/_reduction.py:51: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))


Epoch [1/10], Loss: 21.2510
Epoch [2/10], Loss: 13.8395
Epoch [3/10], Loss: 13.7829
Epoch [4/10], Loss: 13.2680
Epoch [5/10], Loss: 13.0272
Epoch [6/10], Loss: 12.4177
Epoch [7/10], Loss: 12.5217
Epoch [8/10], Loss: 11.7651
Epoch [9/10], Loss: 11.6999
Epoch [10/10], Loss: 11.1090
Epoch [11/10], Loss: 10.5076
Epoch [12/10], Loss: 10.4190
Epoch [13/10], Loss: 10.3078
Epoch [14/10], Loss: 9.9431
Epoch [15/10], Loss: 9.2364
Epoch [16/10], Loss: 9.2677
Epoch [17/10], Loss: 8.8628
Epoch [18/10], Loss: 8.8040
Epoch [19/10], Loss: 8.4975
Epoch [20/10], Loss: 8.4226
Model saved to output/naimishnet.pth
